
# 1. Business Understanding

## 1.1 Project Background

The project focuses on the Brazilian Olist e-commerce marketplace.

The objective is to understand the relationship between delivery
performance, customer satisfaction and sales.

The analysis uses historical Olist e-commerce data from
January 2017 to August 2018.

## 1.2 Business Problem

Olist has data distributed across multiple source files.
There is no single trusted view of delivery performance,
customer satisfaction and sales.

The relationship between late deliveries and customer
review scores needs to be investigated.

## 1.3 Main Research Question

How is delivery performance associated with customer satisfaction and sales outcomes in the Olist e-commerce marketplace?

        - The analysis describes relationships in historical observational data. 
        - It does not by itself prove that delivery delays cause changes in reviews or sales.

## 1.4 Supporting Business Questions

1. How many orders fall within the agreed analysis period, and how does order volume vary by status and month?
2. What is the GMV (item price excluding freight), and what is the average order value (payment value divided by eligible order count)?
3. What proportion of delivered orders arrive on or before the promised date?
4. How long does delivery take from purchase to customer receipt?
5. How many days late are orders that arrive after the promised date?
6. What proportion of eligible orders meet the seller dispatch deadline?
7. What is the average difference between the promised and actual delivery dates?
8. What is the average customer review score?
9. What proportion of reviews are poor (1–2 stars)?
10. How different are average review scores for on-time and late deliveries?
11. What proportion of unique customers place two or more orders in the analysis period?

    Monthly, state and product-category breakdowns can be used as analysis dimensions where the data and sample sizes support them.

## 1.5 Analysis Objectives

- Measure delivery performance.
- Identify delivery delays and patterns.
- Analyse customer review scores.
- Examine the relationship between delivery performance
  and customer satisfaction.
- Compare sales indicators across delivery groups.
- Develop useful business insights.

## 1.6 Project Scope

Dataset: Brazilian Olist e-commerce dataset.

Analysis period: January 2017 to August 2018.

Tools: Python, MySQL and Power BI.

Source data: Nine original CSV files.



## 1.7 Initial KPI Framework

| ID | KPI | Formula supplied by Aimen | Reported baseline | Proposed target |
|---|---|---|---:|---|
| K01 | GMV | SUM(item price), excluding freight | 13,449,529.68 BRL | +10% YoY |
| K02 | Number of orders | Distinct order count, excluding cancelled/unavailable | 97,905 | +10% YoY |
| K03 | Average order value | SUM(payment value) / eligible order count | 160.22 BRL | >= 165 BRL |
| K04 | On-time delivery rate | Delivered on or before estimate / all delivered orders | 93.21% | >= 95% |
| K05 | Average delivery time | Average actual delivery date minus purchase timestamp | 12.54 days | <= 10 days |
| K06 | Average delay of late orders | Average actual delivery date minus estimated date, for late orders | 10.62 days | <= 7 days |
| K07 | Seller dispatch compliance | Orders handed to carrier by shipping deadline / delivered orders | 91.08% | >= 95% |
| K08 | Delivery estimate buffer | Average estimated date minus actual delivery date | 11.81 days | 5–8 days |
| K09 | Average review score | Average review score using latest review per order | 4.09 | >= 4.2 |
| K10 | Poor review rate | Reviews with score <= 2 / all reviews | 14.63% | <= 12% |
| K11 | Review gap: on-time vs late | Average on-time score minus average late score | 2.02 stars | Monitor |
| K12 | Repeat customer rate | Unique customers with 2+ orders / all unique customers | 3.11% | >= 5% |

**Validation note:** Reported KPI baselines are provisional
until independently recalculated. The calculation rules for
K07 and the review population for K09–K11 will be confirmed
before final reporting.

# 2. Data Loading

## 2.1 Objective

The objective of this stage is to load the nine original Olist
CSV files into Python using Pandas.

The datasets will be checked for file availability, successful
loading, row counts, column counts and basic structure.

The original source files will remain unchanged.

In [2762]:
from pathlib import Path

# Find the Olist project folder
current_folder = Path.cwd().resolve()

PROJECT_ROOT = None

for folder in [current_folder, *current_folder.parents]:
    if (folder / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = folder
        break

# Fallback to the known project location
if PROJECT_ROOT is None:
    possible_path = (
        Path.home()
        / "Documents/University/BI_BigData/Olist_Project"
    )

    if (possible_path / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = possible_path

if PROJECT_ROOT is None:
    raise FileNotFoundError("Olist project folder not found.")

# Define the data location
DATA_DIR = PROJECT_ROOT / "02_data_acquisition" / "raw_data"

print("Project location:", PROJECT_ROOT)
print("Data location:", DATA_DIR)
print("Data folder exists:", DATA_DIR.exists())

Project location: /Users/tarun/Documents/University/BI_BigData/Olist_Project
Data location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/02_data_acquisition/raw_data
Data folder exists: True


## 2.3 Check the available CSV files

In [2763]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## 2.4 Load the nine datasets into Python

In this step, we use Pandas to load all nine CSV files
into memory as separate DataFrames.

Each DataFrame represents one source table.
The original CSV files will remain unchanged.

In [2764]:

import pandas as pd
import numpy as np

# Define the nine source files
DATA_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Create an empty dictionary
data = {}

# Load each CSV file
for dataset_name, filename in DATA_FILES.items():

    file_path = DATA_DIR / filename

    data[dataset_name] = pd.read_csv(
        file_path,
        low_memory=False
    )

    print(f"Successfully loaded: {dataset_name}")

print("\nTotal datasets loaded:", len(data))


Successfully loaded: customers
Successfully loaded: geolocation
Successfully loaded: order_items
Successfully loaded: order_payments
Successfully loaded: order_reviews
Successfully loaded: orders
Successfully loaded: products
Successfully loaded: sellers
Successfully loaded: category_translation

Total datasets loaded: 9


## 2.5 Dataset Overview

This section summarises the number of records and
columns in each loaded dataset.

In [2765]:
dataset_summary = []

for name, df in data.items():

    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

dataset_summary_df = pd.DataFrame(dataset_summary)

display(dataset_summary_df)

,Dataset,Rows,Columns,Memory (MB)
0,customers,99441,5,26.59
1,geolocation,1000163,5,129.38
2,order_items,112650,7,35.99
3,order_payments,103886,5,16.23
4,order_reviews,99224,7,39.12
5,orders,99441,8,52.94
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


## 2.6 Initial Inspection of the Orders Dataset

The orders dataset contains order information, order status,
purchase timestamps and delivery timestamps.

It will be a key source for analysing delivery performance.

In [2766]:
# Display the first five records
display(data["orders"].head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [2767]:
# Display the column names
print("Columns in the orders dataset:")

for column in data["orders"].columns:
    print(column)

Columns in the orders dataset:
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date


In [2768]:
# Check the data types
data["orders"].dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

## 2.7 Initial Order Status Overview

This section examines the different order statuses
recorded in the Olist orders dataset.

In [2769]:
# Count orders by status
order_status_summary = (
    data["orders"]["order_status"]
    .value_counts(dropna=False)
)

display(order_status_summary)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [2770]:
# Create reports folder
REPORT_DIR = PROJECT_ROOT / "04_eda" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save dataset overview
summary_file = REPORT_DIR / "02_data_loading_summary.csv"

dataset_summary_df.to_csv(summary_file, index=False)

print("Dataset summary successfully saved!")
print("File location:", summary_file)

Dataset summary successfully saved!
File location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/04_eda/reports/02_data_loading_summary.csv


# 3. Initial Data Exploration

## 3.1 Objective

The objective of this stage is to understand the structure
and business relevance of the nine Olist datasets.

We will identify important tables, columns and variables
required for analysing delivery performance, customer
satisfaction and sales.

Detailed data dictionary and data quality profiling
will be coordinated with Aimen's work.

## 3.2 Identify Key Variables for Business Analysis

This section identifies the important variables required
for analysing delivery performance, customer satisfaction
and sales.

The selection is based on the data dictionary prepared
by Aimen and the project's 12 KPIs.

In [2771]:
# List the columns available in each dataset

for name, df in data.items():

    print("\n" + "=" * 60)
    print("DATASET:", name.upper())
    print("=" * 60)

    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])

    print("\nAvailable columns:")

    for column in df.columns:
        print("-", column)


DATASET: CUSTOMERS
Number of rows: 99441
Number of columns: 5

Available columns:
- customer_id
- customer_unique_id
- customer_zip_code_prefix
- customer_city
- customer_state

DATASET: GEOLOCATION
Number of rows: 1000163
Number of columns: 5

Available columns:
- geolocation_zip_code_prefix
- geolocation_lat
- geolocation_lng
- geolocation_city
- geolocation_state

DATASET: ORDER_ITEMS
Number of rows: 112650
Number of columns: 7

Available columns:
- order_id
- order_item_id
- product_id
- seller_id
- shipping_limit_date
- price
- freight_value

DATASET: ORDER_PAYMENTS
Number of rows: 103886
Number of columns: 5

Available columns:
- order_id
- payment_sequential
- payment_type
- payment_installments
- payment_value

DATASET: ORDER_REVIEWS
Number of rows: 99224
Number of columns: 7

Available columns:
- review_id
- order_id
- review_score
- review_comment_title
- review_comment_message
- review_creation_date
- review_answer_timestamp

DATASET: ORDERS
Number of rows: 99441
Number of 

## 3.2 Identify Key Variables for Business Analysis

This section identifies the important variables needed
to investigate delivery performance, customer satisfaction
and sales.

The selection is based on the data dictionary prepared
by Aimen.

In [2772]:

# Select the variables required for our business analysis

key_variables = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],

    "order_payments": [
        "order_id",
        "payment_sequential",
        "payment_value"
    ],

    "order_reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ],

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_state"
    ],

    "sellers": [
        "seller_id",
        "seller_state"
    ],

    "products": [
        "product_id",
        "product_category_name"
    ],

    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],

    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_state"
    ]
}

# Check whether each selected variable exists
variable_check = []

for dataset_name, columns in key_variables.items():

    for column in columns:

        variable_check.append({
            "Dataset": dataset_name,
            "Variable": column,
            "Available": column in data[dataset_name].columns
        })

variable_check_df = pd.DataFrame(variable_check)

display(variable_check_df)

print("Total variables checked:", len(variable_check_df))
print("Missing variables:", (~variable_check_df["Available"]).sum())


,Dataset,Variable,Available
0,orders,order_id,True
1,orders,customer_id,True
2,orders,order_status,True
3,orders,order_purchase_timestamp,True
4,orders,order_delivered_carrier_date,True
5,orders,order_delivered_customer_date,True
6,orders,order_estimated_delivery_date,True
7,order_items,order_id,True
8,order_items,order_item_id,True
9,order_items,product_id,True


Total variables checked: 35
Missing variables: 0


## 3.3 Understand Dataset Relationships

This section describes how the nine Olist datasets
are connected.

The order_id is the main connection between orders,
items, payments and reviews.

Understanding table relationships is important
to prevent duplicated records and incorrect KPI calculations.

In [2773]:

# Document the relationships between the datasets

relationships = [
    ["Customers", "Orders", "customer_id", "One-to-one"],
    ["Orders", "Order Items", "order_id", "One-to-many"],
    ["Orders", "Payments", "order_id", "One-to-many"],
    ["Orders", "Reviews", "order_id", "Zero-to-many"],
    ["Order Items", "Products", "product_id", "Many-to-one"],
    ["Order Items", "Sellers", "seller_id", "Many-to-one"],
    ["Products", "Category Translation",
     "product_category_name", "Many-to-one"],
    ["Customers", "Geolocation",
     "customer_zip_code_prefix", "Non-unique ZIP match"],
    ["Sellers", "Geolocation",
     "seller_zip_code_prefix", "Non-unique ZIP match"]
]

relationships_df = pd.DataFrame(
    relationships,
    columns=[
        "Parent Table",
        "Related Table",
        "Join Column",
        "Relationship"
    ]
)

display(relationships_df)


,Parent Table,Related Table,Join Column,Relationship
0,Customers,Orders,customer_id,One-to-one
1,Orders,Order Items,order_id,One-to-many
2,Orders,Payments,order_id,One-to-many
3,Orders,Reviews,order_id,Zero-to-many
4,Order Items,Products,product_id,Many-to-one
5,Order Items,Sellers,seller_id,Many-to-one
6,Products,Category Translation,product_category_name,Many-to-one
7,Customers,Geolocation,customer_zip_code_prefix,Non-unique ZIP match
8,Sellers,Geolocation,seller_zip_code_prefix,Non-unique ZIP match


## 3.4 Understanding Date and Time Variables

This section examines the important date and timestamp
variables used in delivery performance analysis.

The main focus is purchase time, carrier handover,
actual delivery and estimated delivery dates.

These variables support KPIs K04, K05, K06, K07 and K08.

In [2774]:

# Identify important date and timestamp columns

date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "order_items": [
        "shipping_limit_date"
    ],

    "order_reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

# Inspect the current data types and sample values

for dataset_name, columns in date_columns.items():

    print("\n" + "=" * 60)
    print("DATASET:", dataset_name.upper())
    print("=" * 60)

    df = data[dataset_name]

    for column in columns:

        print("\nColumn:", column)
        print("Data type:", df[column].dtype)
        print("Sample value:", df[column].dropna().head(1).tolist())



DATASET: ORDERS

Column: order_purchase_timestamp
Data type: str
Sample value: ['2017-10-02 10:56:33']

Column: order_approved_at
Data type: str
Sample value: ['2017-10-02 11:07:15']

Column: order_delivered_carrier_date
Data type: str
Sample value: ['2017-10-04 19:55:00']

Column: order_delivered_customer_date
Data type: str
Sample value: ['2017-10-10 21:25:13']

Column: order_estimated_delivery_date
Data type: str
Sample value: ['2017-10-18 00:00:00']

DATASET: ORDER_ITEMS

Column: shipping_limit_date
Data type: str
Sample value: ['2017-09-19 09:45:35']

DATASET: ORDER_REVIEWS

Column: review_creation_date
Data type: str
Sample value: ['2018-01-18 00:00:00']

Column: review_answer_timestamp
Data type: str
Sample value: ['2018-01-18 21:46:59']


## 3.5 Convert Date and Time Columns

Convert the relevant text columns into proper datetime
formats for date calculations and delivery analysis.

The original source CSV files will remain unchanged.

In [2775]:

# Create working copies of the original datasets

analysis_data = {
    name: df.copy()
    for name, df in data.items()
}

# Convert the selected date columns
date_conversion_results = []

for dataset_name, columns in date_columns.items():

    for column in columns:

        original_values = analysis_data[dataset_name][column]

        converted_values = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        # Count non-empty values that could not be converted
        invalid_values = (
            original_values.notna() &
            converted_values.isna()
        )

        analysis_data[dataset_name][column] = converted_values

        date_conversion_results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Original Missing": original_values.isna().sum(),
            "Invalid Dates": invalid_values.sum(),
            "New Data Type": str(
                analysis_data[dataset_name][column].dtype
            )
        })

date_conversion_summary = pd.DataFrame(
    date_conversion_results
)

display(date_conversion_summary)


,Dataset,Column,Original Missing,Invalid Dates,New Data Type
0,orders,order_purchase_timestamp,0,0,datetime64[us]
1,orders,order_approved_at,160,0,datetime64[us]
2,orders,order_delivered_carrier_date,1783,0,datetime64[us]
3,orders,order_delivered_customer_date,2965,0,datetime64[us]
4,orders,order_estimated_delivery_date,0,0,datetime64[us]
5,order_items,shipping_limit_date,0,0,datetime64[us]
6,order_reviews,review_creation_date,0,0,datetime64[us]
7,order_reviews,review_answer_timestamp,0,0,datetime64[us]


In [2776]:
# Check converted date columns

for dataset_name, columns in date_columns.items():

    print("\nDataset:", dataset_name)

    for column in columns:
        print(
            column,
            "->",
            analysis_data[dataset_name][column].dtype
        )


Dataset: orders
order_purchase_timestamp -> datetime64[us]
order_approved_at -> datetime64[us]
order_delivered_carrier_date -> datetime64[us]
order_delivered_customer_date -> datetime64[us]
order_estimated_delivery_date -> datetime64[us]

Dataset: order_items
shipping_limit_date -> datetime64[us]

Dataset: order_reviews
review_creation_date -> datetime64[us]
review_answer_timestamp -> datetime64[us]


## 3.6 Apply the Project Analysis Period

The project analysis period is 1 January 2017 to
31 August 2018, inclusive.

The order_purchase_timestamp field will be used
to identify orders within this period.

The original orders dataset will remain unchanged.

In [2777]:

# Define the project period
start_date = pd.Timestamp("2017-01-01")
end_date_exclusive = pd.Timestamp("2018-09-01")

# Create a working copy of the orders dataset
orders_df = analysis_data["orders"].copy()

# Select orders within the agreed period
in_scope_mask = (
    orders_df["order_purchase_timestamp"].notna()
    & (orders_df["order_purchase_timestamp"] >= start_date)
    & (orders_df["order_purchase_timestamp"] < end_date_exclusive)
)

# Create separate datasets
scoped_orders = orders_df.loc[in_scope_mask].copy()
out_of_scope_orders = orders_df.loc[~in_scope_mask].copy()

# Display the results
print("Total original orders:", len(orders_df))
print("Orders within project period:", len(scoped_orders))
print("Orders outside project period:", len(out_of_scope_orders))

print("\nFirst purchase date:",
      scoped_orders["order_purchase_timestamp"].min())

print("Last purchase date:",
      scoped_orders["order_purchase_timestamp"].max())


Total original orders: 99441
Orders within project period: 99092
Orders outside project period: 349

First purchase date: 2017-01-05 11:56:06
Last purchase date: 2018-08-31 16:13:44


## 3.7 Define the Eligible Order Population (K02)

The project-period population contains 99,092 orders.

For K02, cancelled and unavailable orders are excluded.

The eligible order population will be checked against
Aimen's reported figure of 97,905 orders.

In [2778]:
# Check order statuses within the project period

status_summary = (
    scoped_orders["order_status"]
    .value_counts(dropna=False)
)

display(status_summary)

order_status
delivered      96211
shipped         1097
unavailable      602
canceled         580
processing       299
invoiced         296
created            5
approved           2
Name: count, dtype: int64

In [2779]:
# Exclude cancelled and unavailable orders

excluded_statuses = ["canceled", "unavailable"]

eligible_orders = scoped_orders.loc[
    ~scoped_orders["order_status"].isin(excluded_statuses)
].copy()

# Count unique eligible orders
k02_order_count = eligible_orders["order_id"].nunique()

print("Project-period orders:", len(scoped_orders))
print("Excluded orders:", len(scoped_orders) - len(eligible_orders))
print("Eligible orders (K02):", k02_order_count)

Project-period orders: 99092
Excluded orders: 1182
Eligible orders (K02): 97910


In [2780]:
# Method 1: Exclude cancelled and unavailable
eligible_v1 = scoped_orders[
    ~scoped_orders["order_status"].isin(
        ["canceled", "unavailable"]
    )
]

# Method 2: Also exclude created orders
eligible_v2 = scoped_orders[
    ~scoped_orders["order_status"].isin(
        ["canceled", "unavailable", "created"]
    )
]

print("Method 1 - Exclude cancelled and unavailable:")
print("Orders:", eligible_v1["order_id"].nunique())

print("\nMethod 2 - Also exclude created:")
print("Orders:", eligible_v2["order_id"].nunique())

Method 1 - Exclude cancelled and unavailable:
Orders: 97910

Method 2 - Also exclude created:
Orders: 97905


## 3.8 Define the Delivery Analysis Population

Delivery performance KPIs require successfully delivered
orders with a valid actual customer delivery date.

Orders without an actual delivery date will not be included
in delivery time and delivery delay calculations.

In [2781]:
# Select orders marked as delivered
delivered_candidates = scoped_orders[
    scoped_orders["order_status"] == "delivered"
].copy()

# Exclude delivered orders without an actual delivery date
delivery_orders = delivered_candidates[
    delivered_candidates["order_delivered_customer_date"].notna()
].copy()

# Calculate the population
print("Delivered orders:", 
      delivered_candidates["order_id"].nunique())

print("Delivered orders without actual delivery date:",
      delivered_candidates[
          "order_delivered_customer_date"
      ].isna().sum())

print("Valid delivery analysis population:",
      delivery_orders["order_id"].nunique())

Delivered orders: 96211
Delivered orders without actual delivery date: 8
Valid delivery analysis population: 96203


### 3.8 Findings — Delivery Analysis Population

- Orders within project period: 99,092
- Orders marked as delivered: 96,211
- Delivered orders missing actual delivery date: 8
- Valid delivery analysis population: 96,203

The 8 delivered orders without an actual delivery date
cannot be used for delivery-time and lateness calculations.

The original source data remains unchanged.

# 4. Data Preparation

## 4.1 Create Working Copies

Working copies will be created for data preparation and analysis.

The original source data will remain unchanged.

Order-related tables will be restricted to orders within
the agreed project period.

Reference tables will initially be retained in full.

In [2782]:
# Create the project-period order ID list
scope_order_ids = set(scoped_orders["order_id"])

# Create a separate collection of prepared datasets
prepared_data = {}

# 1. Orders within the project period
prepared_data["orders"] = scoped_orders.copy()

# 2. Filter order items to in-scope orders
prepared_data["order_items"] = (
    analysis_data["order_items"][
        analysis_data["order_items"]["order_id"].isin(scope_order_ids)
    ].copy()
)

# 3. Filter payments to in-scope orders
prepared_data["order_payments"] = (
    analysis_data["order_payments"][
        analysis_data["order_payments"]["order_id"].isin(scope_order_ids)
    ].copy()
)

# 4. Filter reviews to in-scope orders
# Do not remove duplicate reviews yet.
prepared_data["order_reviews"] = (
    analysis_data["order_reviews"][
        analysis_data["order_reviews"]["order_id"].isin(scope_order_ids)
    ].copy()
)

# 5. Retain reference datasets for later joins and validation
reference_tables = [
    "customers",
    "sellers",
    "products",
    "category_translation",
    "geolocation"
]

for table in reference_tables:
    prepared_data[table] = analysis_data[table].copy()

print("Working datasets created successfully.")
print("Total datasets:", len(prepared_data))

Working datasets created successfully.
Total datasets: 9


In [2783]:
# Display the prepared dataset overview

prepared_summary = []

for name, df in prepared_data.items():

    prepared_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

prepared_summary_df = pd.DataFrame(prepared_summary)

display(prepared_summary_df)

print("Total prepared datasets:", len(prepared_data))

,Dataset,Rows,Columns
0,orders,99092,8
1,order_items,112279,7
2,order_payments,103520,5
3,order_reviews,98879,7
4,customers,99441,5
5,sellers,3095,4
6,products,32951,9
7,category_translation,71,2
8,geolocation,1000163,5


Total prepared datasets: 9


## 4.2 Validate Project-Period Filtering

This section compares the original order-related tables
with their prepared versions.

The objective is to confirm that only records linked
to orders within the agreed project period are retained.

In [2784]:
# Validate the scope filtering

filter_validation = []

for table in ["orders", "order_items", "order_payments", "order_reviews"]:

    original_df = analysis_data[table]
    prepared_df = prepared_data[table]

    filter_validation.append({
        "Dataset": table,
        "Original Rows": len(original_df),
        "Prepared Rows": len(prepared_df),
        "Rows Excluded": len(original_df) - len(prepared_df),
        "Prepared Rows with Out-of-Scope Order IDs": (
            ~prepared_df["order_id"].isin(scope_order_ids)
        ).sum()
    })

filter_validation_df = pd.DataFrame(filter_validation)

display(filter_validation_df)

,Dataset,Original Rows,Prepared Rows,Rows Excluded,Prepared Rows with Out-of-Scope Order IDs
0,orders,99441,99092,349,0
1,order_items,112650,112279,371,0
2,order_payments,103886,103520,366,0
3,order_reviews,99224,98879,345,0


## 4.3 Review Data Preparation

This section examines review records within the project period.

The analysis will investigate multiple reviews per order,
duplicate review IDs and review timestamps.

The latest-review selection rule will be applied only
after the timestamp and tie-breaking rules are confirmed.

In [2785]:
# Create a working copy of the prepared review data

reviews_df = prepared_data["order_reviews"].copy()

# Number of reviews per order
reviews_per_order = reviews_df.groupby("order_id").size()

# Review data summary
print("Total review records:", len(reviews_df))
print("Unique reviewed orders:", reviews_df["order_id"].nunique())

print(
    "Duplicate review ID occurrences:",
    reviews_df.duplicated(subset=["review_id"]).sum()
)

print(
    "Orders with multiple reviews:",
    (reviews_per_order > 1).sum()
)

print(
    "Maximum reviews for one order:",
    reviews_per_order.max()
)

Total review records: 98879
Unique reviewed orders: 98330
Duplicate review ID occurrences: 797
Orders with multiple reviews: 545
Maximum reviews for one order: 3


In [2786]:
# Check timestamp completeness
print("Missing review creation dates:",
      reviews_df["review_creation_date"].isna().sum())

print("Missing review answer timestamps:",
      reviews_df["review_answer_timestamp"].isna().sum())

# Check whether the latest answer timestamp is tied
latest_answer = reviews_df.groupby(
    "order_id"
)["review_answer_timestamp"].transform("max")

latest_answer_rows = reviews_df[
    reviews_df["review_answer_timestamp"] == latest_answer
]

answer_ties = (
    latest_answer_rows.groupby("order_id").size() > 1
).sum()

print("Orders with tied latest answer timestamps:", answer_ties)

Missing review creation dates: 0
Missing review answer timestamps: 0
Orders with tied latest answer timestamps: 0


In [2787]:
# Review population overview

print("Total review records:", len(reviews_df))

print(
    "Unique reviewed orders:",
    reviews_df["order_id"].nunique()
)

print(
    "Duplicate review ID occurrences:",
    reviews_df.duplicated(subset=["review_id"]).sum()
)

reviews_per_order = reviews_df.groupby("order_id").size()

print(
    "Orders with multiple reviews:",
    (reviews_per_order > 1).sum()
)

print(
    "Maximum reviews per order:",
    reviews_per_order.max()
)

Total review records: 98879
Unique reviewed orders: 98330
Duplicate review ID occurrences: 797
Orders with multiple reviews: 545
Maximum reviews per order: 3


### 4.3.1 Investigate Duplicate Review Records

Examine duplicate review IDs and multiple reviews per order.

Identify exact duplicate rows and understand how repeated
review records are distributed before applying a selection rule.

In [2788]:
# Check exact duplicate records
print(
    "Exact duplicate rows:",
    reviews_df.duplicated().sum()
)

# Check duplicate review ID and order ID combinations
print(
    "Duplicate review_id + order_id combinations:",
    reviews_df.duplicated(
        subset=["review_id", "order_id"]
    ).sum()
)

# Count repeated review ID groups
duplicate_ids = reviews_df[
    reviews_df.duplicated("review_id", keep=False)
]

print(
    "Number of repeated review ID groups:",
    duplicate_ids["review_id"].nunique()
)

# Show the distribution of reviews per order
review_distribution = (
    reviews_per_order.value_counts()
    .sort_index()
    .rename_axis("Reviews per order")
    .to_frame("Number of orders")
)

display(review_distribution)

Exact duplicate rows: 0
Duplicate review_id + order_id combinations: 0
Number of repeated review ID groups: 776


,Number of orders
Reviews per order,
1,97785
2,541
3,4


In [2789]:
# Display examples of repeated review IDs

duplicate_examples = (
    duplicate_ids.sort_values(
        ["review_id", "review_answer_timestamp"]
    )
)

display(
    duplicate_examples[
        [
            "review_id",
            "order_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp"
        ]
    ].head(12)
)

,review_id,order_id,review_score,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,2018-03-07,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,2018-03-07,2018-03-20 18:08:23
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,2017-09-21,2017-09-26 03:27:47
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,2017-09-21,2017-09-26 03:27:47
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,2018-03-07,2018-03-08 03:00:53
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,2017-09-09,2017-09-13 09:52:44


### 4.3.2 Select the Latest Review per Order

The review dataset contains multiple reviews for some orders.

For K09, K10 and K11, the latest review per order
will be selected using review_answer_timestamp.

The original review records will remain unchanged.

In [2790]:
# Sort reviews by order and answer timestamp
reviews_sorted = reviews_df.sort_values(
    ["order_id", "review_answer_timestamp"]
)

# Keep the latest review for each order
latest_reviews = (
    reviews_sorted
    .drop_duplicates(subset="order_id", keep="last")
    .copy()
)

print("Original review records:", len(reviews_df))
print("Prepared latest reviews:", len(latest_reviews))
print("Unique orders:", latest_reviews["order_id"].nunique())
print(
    "Duplicate order IDs:",
    latest_reviews["order_id"].duplicated().sum()
)

Original review records: 98879
Prepared latest reviews: 98330
Unique orders: 98330
Duplicate order IDs: 0


In [2791]:
review_preparation_summary = pd.DataFrame([
    {
        "Measurement": "Original review records",
        "Count": len(reviews_df)
    },
    {
        "Measurement": "Unique reviewed orders",
        "Count": reviews_df["order_id"].nunique()
    },
    {
        "Measurement": "Orders with multiple reviews",
        "Count": (reviews_per_order > 1).sum()
    },
    {
        "Measurement": "Prepared latest reviews",
        "Count": len(latest_reviews)
    },
    {
        "Measurement": "Records not selected",
        "Count": len(reviews_df) - len(latest_reviews)
    }
])

display(review_preparation_summary)

,Measurement,Count
0,Original review records,98879
1,Unique reviewed orders,98330
2,Orders with multiple reviews,545
3,Prepared latest reviews,98330
4,Records not selected,549


### 4.4 Seller Dispatch Data Preparation (K07)

Seller dispatch compliance measures whether orders
are handed over to the carrier within the shipping deadline.

The order-items table contains shipping deadlines,
while the orders table contains the carrier handover date.

Multiple item deadlines per order must be examined
before defining an order-level compliance rule.

In [2792]:
# Create a working copy of order items
items_df = prepared_data["order_items"].copy()

# Count different shipping deadlines per order
deadlines_per_order = (
    items_df.groupby("order_id")["shipping_limit_date"]
    .nunique()
)

print(
    "Orders with multiple shipping deadlines:",
    (deadlines_per_order > 1).sum()
)

print(
    "Maximum different deadlines per order:",
    deadlines_per_order.max()
)

print(
    "Orders with one shipping deadline:",
    (deadlines_per_order == 1).sum()
)

Orders with multiple shipping deadlines: 334
Maximum different deadlines per order: 3
Orders with one shipping deadline: 98019


In [2793]:
# Find orders with multiple shipping deadlines
multi_deadline_ids = deadlines_per_order[
    deadlines_per_order > 1
].index

# Display 10 examples
deadline_examples = items_df[
    items_df["order_id"].isin(multi_deadline_ids)
].sort_values(["order_id", "shipping_limit_date"])

display(
    deadline_examples[
        [
            "order_id",
            "order_item_id",
            "shipping_limit_date",
            "seller_id"
        ]
    ].head(20)
)

,order_id,order_item_id,shipping_limit_date,seller_id
789,01cce1175ac3c4a450e3a0f856d02734,2,2018-07-23 20:43:14,0bae85eb84b9fb3bd773911e89288d54
788,01cce1175ac3c4a450e3a0f856d02734,1,2018-07-25 20:43:14,9c7fd21e0aa159b20da593371f08ff7c
1079,0282e83acfe3e7aaf60c871a4a9aca61,1,2018-05-16 20:57:02,db4350fd57ae30082dec7acbaacc17f9
1080,0282e83acfe3e7aaf60c871a4a9aca61,2,2018-05-17 20:57:02,2a6fbeaa7ab9e5d59a35841a8da71ce4
1201,02c9fa29186a8d00b3e73d3837ea40e5,2,2018-08-21 19:09:45,6061155addc1e54b4cfb51c1c2a32ad8
1200,02c9fa29186a8d00b3e73d3837ea40e5,1,2018-08-22 19:09:45,08ad4ac1388e4420ca531c3edfc46198
1249,02e405aa2667a116637cdb4affaf93c0,1,2018-04-23 23:51:37,640e21a7d01df7614a3b4923e990d40c
1250,02e405aa2667a116637cdb4affaf93c0,2,2018-04-26 23:51:37,391fc6631aebcf3004804e51b40bcf1e
1713,03ecec245220b63fd7f68c1737ba99ba,1,2018-06-13 20:23:08,fe2032dab1a61af8794248c8196565c9
1714,03ecec245220b63fd7f68c1737ba99ba,2,2018-06-15 20:23:08,620c87c171fb2a6dd6e8bb4dec959fc6


### 4.4.1 Aggregate Shipping Deadlines by Order

Each order may contain multiple items with different
shipping deadlines.

For order-level seller dispatch compliance, we will
use the earliest item shipping deadline.

This is a conservative order-level proxy because
the carrier handover date is recorded at order level.

In [2794]:
# Aggregate item deadlines at order level

order_deadlines = (
    items_df.groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        distinct_deadlines=("shipping_limit_date", "nunique"),
        earliest_deadline=("shipping_limit_date", "min"),
        latest_deadline=("shipping_limit_date", "max"),
        distinct_sellers=("seller_id", "nunique")
    )
    .reset_index()
)

display(order_deadlines.head(10))

print("Orders with item records:", len(order_deadlines))

print(
    "Orders with multiple deadlines:",
    (order_deadlines["distinct_deadlines"] > 1).sum()
)

,order_id,item_count,distinct_deadlines,earliest_deadline,latest_deadline,distinct_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,2017-09-19 09:45:35,2017-09-19 09:45:35,1
1,00018f77f2f0320c557190d7a144bdd3,1,1,2017-05-03 11:05:13,2017-05-03 11:05:13,1
2,000229ec398224ef6ca0657da4fc703e,1,1,2018-01-18 14:48:30,2018-01-18 14:48:30,1
3,00024acbcdf0a6daa1e931b038114c75,1,1,2018-08-15 10:10:18,2018-08-15 10:10:18,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,2017-02-13 13:57:51,2017-02-13 13:57:51,1
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,1,2017-05-23 03:55:27,2017-05-23 03:55:27,1
6,00054e8431b9d7675808bcb819fb4a32,1,1,2017-12-14 12:10:31,2017-12-14 12:10:31,1
7,000576fe39319847cbb9d288c5617fa6,1,1,2018-07-10 12:30:45,2018-07-10 12:30:45,1
8,0005a1a1728c9d785b8e2b08b904576c,1,1,2018-03-26 18:31:29,2018-03-26 18:31:29,1
9,0005f50442cb953dcd1d21e1fb923495,1,1,2018-07-06 14:10:56,2018-07-06 14:10:56,1


Orders with item records: 98353
Orders with multiple deadlines: 334


In [2795]:
# Check carrier handover dates for delivery orders

print("Valid delivery orders:", len(delivery_orders))

print(
    "Missing carrier handover dates:",
    delivery_orders["order_delivered_carrier_date"].isna().sum()
)

Valid delivery orders: 96203
Missing carrier handover dates: 1


### 4.4.2 Match Delivery Dates with Shipping Deadlines

Combine the delivered orders with their aggregated
shipping deadlines.

Check missing matches, missing carrier handover dates
and orders involving multiple sellers.

In [2796]:
# Select the required delivery fields
delivery_fields = delivery_orders[
    [
        "order_id",
        "order_delivered_carrier_date"
    ]
].copy()

# Select the aggregated shipping deadline fields
deadline_fields = order_deadlines[
    [
        "order_id",
        "item_count",
        "distinct_deadlines",
        "earliest_deadline",
        "latest_deadline",
        "distinct_sellers"
    ]
].copy()

# Merge delivery information with shipping deadlines
dispatch_check = delivery_fields.merge(
    deadline_fields,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

# Display matching results
print("Total delivered orders:", len(dispatch_check))

print(
    "Orders matched with item deadlines:",
    (dispatch_check["_merge"] == "both").sum()
)

print(
    "Orders without matching item records:",
    (dispatch_check["_merge"] == "left_only").sum()
)

print(
    "Missing carrier handover dates:",
    dispatch_check["order_delivered_carrier_date"].isna().sum()
)

print(
    "Orders involving multiple sellers:",
    (dispatch_check["distinct_sellers"] > 1).sum()
)

Total delivered orders: 96203
Orders matched with item deadlines: 96203
Orders without matching item records: 0
Missing carrier handover dates: 1
Orders involving multiple sellers: 1272


### 4.4.3 Calculate Seller Dispatch Compliance (K07)

The earliest item shipping deadline is used as the
order-level deadline.

An order is considered compliant when the carrier
handover date is on or before this deadline.

Orders with missing carrier handover dates are excluded
from the calculation and reported separately.

This is an order-level proxy, not an individual
seller performance measure.

In [2797]:

# Prepare the K07 calculation population

dispatch_kpi_df = dispatch_check[
    (dispatch_check["_merge"] == "both") &
    (dispatch_check["order_delivered_carrier_date"].notna()) &
    (dispatch_check["earliest_deadline"].notna())
].copy()

# Calculate dispatch compliance
dispatch_kpi_df["dispatch_compliant"] = (
    dispatch_kpi_df["order_delivered_carrier_date"]
    <= dispatch_kpi_df["earliest_deadline"]
)

# Calculate K07
k07_total_orders = dispatch_kpi_df["order_id"].nunique()

k07_compliant_orders = (
    dispatch_kpi_df.loc[
        dispatch_kpi_df["dispatch_compliant"],
        "order_id"
    ].nunique()
)

k07_rate = (k07_compliant_orders / k07_total_orders) * 100

print("Valid K07 orders:", k07_total_orders)
print("Compliant orders:", k07_compliant_orders)
print("Non-compliant orders:", k07_total_orders - k07_compliant_orders)
print(f"Seller dispatch compliance: {k07_rate:.2f}%")


Valid K07 orders: 96202
Compliant orders: 87610
Non-compliant orders: 8592
Seller dispatch compliance: 91.07%


In [2798]:
# Compare dispatch compliance by number of sellers

dispatch_kpi_df["Seller Group"] = np.where(
    dispatch_kpi_df["distinct_sellers"] > 1,
    "Multiple sellers",
    "Single seller"
)

dispatch_comparison = (
    dispatch_kpi_df.groupby("Seller Group")
    .agg(
        Orders=("order_id", "nunique"),
        Compliant_Orders=("dispatch_compliant", "sum")
    )
)

dispatch_comparison["Compliance_Rate_%"] = (
    dispatch_comparison["Compliant_Orders"]
    / dispatch_comparison["Orders"] * 100
)

display(dispatch_comparison)

,Orders,Compliant_Orders,Compliance_Rate_%
Seller Group,,,
Multiple sellers,1272,1210,95.125786
Single seller,94930,86400,91.014432


### 4.4.4 K07 — Seller Dispatch Compliance Results

Calculation rule:
Use the earliest item shipping deadline for each order.
An order is compliant when carrier handover occurs
on or before that deadline.

Results:
- Valid K07 orders: 96,202
- Compliant orders: 87,610
- Non-compliant orders: 8,592
- Calculated compliance rate: 91.07%

Comparison:
- Single-seller orders: 91.01%
- Multiple-seller orders: 95.13%

One delivered order was excluded because its carrier
handover date was missing.

Limitation:
This is an order-level dispatch compliance proxy.
It does not measure individual seller performance.

Aimen's reported baseline: 91.08%.
The difference will be reviewed before final reporting.

### 4.5 Product Category Preparation

Examine missing product categories and missing English
category translations.

The objective is to prepare consistent category information
for product-level and sales analysis.

Original product and translation data will remain unchanged.

In [2799]:
# Create working copies
products_df = prepared_data["products"].copy()
translation_df = prepared_data["category_translation"].copy()

# Check missing product categories
print(
    "Products without category:",
    products_df["product_category_name"].isna().sum()
)

# Check categories without English translation
known_categories = set(
    translation_df["product_category_name"]
)

product_categories = set(
    products_df["product_category_name"].dropna()
)

missing_translations = product_categories - known_categories

print(
    "Category names without English translation:",
    len(missing_translations)
)

print("\nMissing translation examples:")
print(sorted(missing_translations)[:20])

Products without category: 610
Category names without English translation: 2

Missing translation examples:
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [2800]:
# Identify categories without English translation

unmapped_categories = sorted(
    product_categories - known_categories
)

unmapped_products = products_df[
    products_df["product_category_name"].isin(
        unmapped_categories
    )
]

# Count products affected by each category
unmapped_summary = (
    unmapped_products
    .groupby("product_category_name")
    .size()
    .reset_index(name="Product Count")
)

display(unmapped_summary)

print("Products with missing categories:",
      products_df["product_category_name"].isna().sum())

print("Products with untranslated categories:",
      len(unmapped_products))

print("Total affected products:",
      products_df["product_category_name"].isna().sum()
      + len(unmapped_products))

,product_category_name,Product Count
0,pc_gamer,3
1,portateis_cozinha_e_preparadores_de_alimentos,10


Products with missing categories: 610
Products with untranslated categories: 13
Total affected products: 623


### 4.5.2 Prepare Product Categories

Prepare a working product dataset for category analysis.

Missing categories will be labelled as unknown.

Existing English translations will be used.
For categories without translations, the original
Portuguese category will be retained until verified.

Original product data will remain unchanged.

In [2801]:
# Check whether the translation key is unique

print(
    "Duplicate category keys:",
    translation_df["product_category_name"].duplicated().sum()
)

Duplicate category keys: 0


In [2802]:
# Create a separate working copy
products_prepared = products_df.copy()

# Remember which products originally had missing categories
products_prepared["category_was_missing"] = (
    products_prepared["product_category_name"].isna()
)

# Label missing categories
products_prepared["product_category_name"] = (
    products_prepared["product_category_name"].fillna("unknown")
)

# Join English category translations
products_prepared = products_prepared.merge(
    translation_df,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

# Create a consistent analysis label
products_prepared["analysis_category"] = (
    products_prepared["product_category_name_english"]
    .fillna(products_prepared["product_category_name"])
)

# Record translation status
products_prepared["translation_status"] = np.select(
    [
        products_prepared["category_was_missing"],
        products_prepared["product_category_name_english"].notna()
    ],
    [
        "Missing category",
        "English translation available"
    ],
    default="Portuguese category retained"
)

print("Product preparation completed.")
print("Original product records:", len(products_df))
print("Prepared product records:", len(products_prepared))
print(
    "Missing analysis categories:",
    products_prepared["analysis_category"].isna().sum()
)

Product preparation completed.
Original product records: 32951
Prepared product records: 32951
Missing analysis categories: 0


In [2803]:
category_status = (
    products_prepared["translation_status"]
    .value_counts()
)

display(category_status)

print(
    "Unique analysis categories:",
    products_prepared["analysis_category"].nunique()
)

translation_status
English translation available    32328
Missing category                   610
Portuguese category retained        13
Name: count, dtype: int64

Unique analysis categories: 74


### 4.6 Geolocation Data Preparation

The geolocation dataset provides latitude, longitude,
city and state information using ZIP code prefixes.

This section investigates duplicate records, missing
ZIP prefixes, geographic coordinates and ZIP prefix
relationships.

The objective is to prepare a reliable geolocation
reference table for regional analysis.

Original source data will remain unchanged.

In [2804]:
# Create a working copy of geolocation data

geo_df = prepared_data["geolocation"].copy()

# Basic overview
print("Total geolocation records:", len(geo_df))

print(
    "Unique ZIP prefixes:",
    geo_df["geolocation_zip_code_prefix"].nunique()
)

print(
    "Exact duplicate rows:",
    geo_df.duplicated().sum()
)

print(
    "Missing ZIP prefixes:",
    geo_df["geolocation_zip_code_prefix"].isna().sum()
)

print(
    "Missing latitude values:",
    geo_df["geolocation_lat"].isna().sum()
)

print(
    "Missing longitude values:",
    geo_df["geolocation_lng"].isna().sum()
)

print("\nColumn data types:")
print(geo_df.dtypes)

print("\nSample records:")
display(geo_df.head())

Total geolocation records: 1000163
Unique ZIP prefixes: 19015
Exact duplicate rows: 261831
Missing ZIP prefixes: 0
Missing latitude values: 0
Missing longitude values: 0

Column data types:
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Sample records:


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [2805]:
# Identify records with coordinates available

coordinates_present = (
    geo_df["geolocation_lat"].notna()
    & geo_df["geolocation_lng"].notna()
)

# Identify points outside the documented Brazil bounds
outside_brazil = coordinates_present & (
    ~geo_df["geolocation_lat"].between(-33.8, 5.3)
    | ~geo_df["geolocation_lng"].between(-74.0, -34.8)
)

print(
    "Records with coordinates:",
    coordinates_present.sum()
)

print(
    "Points outside Brazil bounds:",
    outside_brazil.sum()
)

print(
    "Records with missing coordinates:",
    (~coordinates_present).sum()
)

Records with coordinates: 1000163
Points outside Brazil bounds: 47
Records with missing coordinates: 0


In [2806]:
# Count different states associated with each ZIP prefix

states_per_zip = (
    geo_df.groupby("geolocation_zip_code_prefix")[
        "geolocation_state"
    ].nunique()
)

multiple_state_zips = states_per_zip[states_per_zip > 1]

print(
    "ZIP prefixes linked to multiple states:",
    len(multiple_state_zips)
)

ZIP prefixes linked to multiple states: 8


In [2807]:
# Check exact duplicate geolocation records

print("Original geolocation records:", len(geo_df))

print(
    "Exact duplicate rows:",
    geo_df.duplicated().sum()
)

print(
    "Unique ZIP prefixes:",
    geo_df["geolocation_zip_code_prefix"].nunique()
)

Original geolocation records: 1000163
Exact duplicate rows: 261831
Unique ZIP prefixes: 19015


In [2808]:
# Identify ZIP prefixes associated with multiple states

problem_zip_prefixes = multiple_state_zips.index

problem_zip_details = (
    geo_df[
        geo_df["geolocation_zip_code_prefix"].isin(
            problem_zip_prefixes
        )
    ]
    .groupby("geolocation_zip_code_prefix")
    .agg(
        States=("geolocation_state",
                lambda x: ", ".join(sorted(x.unique()))),
        Number_of_States=("geolocation_state", "nunique"),
        Number_of_Records=("geolocation_state", "size")
    )
    .reset_index()
)

display(problem_zip_details)

,geolocation_zip_code_prefix,States,Number_of_States,Number_of_Records
0,2116,"RN, SP",2,13
1,4011,"AC, SP",2,179
2,21550,"AC, RJ",2,171
3,23056,"AC, RJ",2,61
4,72915,"DF, GO",2,41
5,78557,"MT, RO",2,97
6,79750,"MS, RS",2,180
7,80630,"PR, SC",2,123


### 4.6.1 Prepare Geolocation Data

Prepare a working geolocation reference table.

Preparation rules:
- Remove exact duplicate records.
- Exclude coordinates outside the agreed Brazil screening bounds.
- Aggregate latitude and longitude by ZIP prefix and state.
- Use ZIP prefix and state as the matching key.

Original geolocation data will remain unchanged.

In [2809]:
# Create a separate working copy
geo_work = geo_df.copy()

# Standardise ZIP prefixes as five-character text
geo_work["geolocation_zip_code_prefix"] = (
    geo_work["geolocation_zip_code_prefix"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(5)
)

# 1. Remove exact duplicate rows
geo_deduplicated = geo_work.drop_duplicates().copy()

print("Original records:", len(geo_work))
print("After duplicate removal:", len(geo_deduplicated))
print("Duplicate rows removed:",
      len(geo_work) - len(geo_deduplicated))

# 2. Keep coordinates within the agreed Brazil bounds
inside_brazil = (
    geo_deduplicated["geolocation_lat"].between(-33.8, 5.3)
    & geo_deduplicated["geolocation_lng"].between(-74.0, -34.8)
)

geo_brazil = geo_deduplicated.loc[inside_brazil].copy()

print("Records retained within Brazil bounds:", len(geo_brazil))
print("Records excluded:", len(geo_deduplicated) - len(geo_brazil))

Original records: 1000163
After duplicate removal: 738332
Duplicate rows removed: 261831
Records retained within Brazil bounds: 738294
Records excluded: 38


In [2810]:
# Create one representative location per ZIP prefix and state

geolocation_prepared = (
    geo_brazil
    .groupby(
        ["geolocation_zip_code_prefix", "geolocation_state"],
        as_index=False
    )
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean"),
        source_point_count=("geolocation_lat", "count")
    )
)

display(geolocation_prepared.head(10))

print("Prepared geolocation records:",
      len(geolocation_prepared))

print(
    "Duplicate ZIP + state keys:",
    geolocation_prepared.duplicated(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    ).sum()
)

,geolocation_zip_code_prefix,geolocation_state,latitude,longitude,source_point_count
0,01001,SP,-23.550227,-46.634039,11
1,01002,SP,-23.547657,-46.634991,6
2,01003,SP,-23.549000,-46.635582,11
3,01004,SP,-23.549829,-46.634792,14
4,01005,SP,-23.549547,-46.636406,13
5,01006,SP,-23.550127,-46.636044,7
6,01007,SP,-23.549962,-46.637204,16
7,01008,SP,-23.546000,-46.635877,13
8,01009,SP,-23.546891,-46.636498,11
9,01010,SP,-23.546667,-46.635404,6


Prepared geolocation records: 19018
Duplicate ZIP + state keys: 0


In [2811]:
# Final geolocation validation

print("Missing latitude:",
      geolocation_prepared["latitude"].isna().sum())

print("Missing longitude:",
      geolocation_prepared["longitude"].isna().sum())

print(
    "Coordinates outside Brazil bounds:",
    (
        ~geolocation_prepared["latitude"].between(-33.8, 5.3)
        | ~geolocation_prepared["longitude"].between(-74.0, -34.8)
    ).sum()
)

print(
    "Duplicate ZIP + state keys:",
    geolocation_prepared.duplicated(
        ["geolocation_zip_code_prefix", "geolocation_state"]
    ).sum()
)

print(
    "Unique ZIP prefixes:",
    geolocation_prepared["geolocation_zip_code_prefix"].nunique()
)

print(
    "Prepared ZIP + state records:",
    len(geolocation_prepared)
)

Missing latitude: 0
Missing longitude: 0
Coordinates outside Brazil bounds: 0
Duplicate ZIP + state keys: 0
Unique ZIP prefixes: 19010
Prepared ZIP + state records: 19018


### 4.6.2 Validate Customer and Seller Geolocation Matching

Check whether customer and seller ZIP prefixes
and state combinations match the prepared geolocation table.

ZIP prefixes will be standardised as five-character text.

In [2812]:
# Prepare customer location keys
customer_keys = prepared_data["customers"][
    ["customer_zip_code_prefix", "customer_state"]
].copy()

customer_keys.columns = ["zip_prefix", "state"]

# Prepare seller location keys
seller_keys = prepared_data["sellers"][
    ["seller_zip_code_prefix", "seller_state"]
].copy()

seller_keys.columns = ["zip_prefix", "state"]

# Standardise ZIP prefixes
for df in [customer_keys, seller_keys]:
    df["zip_prefix"] = (
        df["zip_prefix"]
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(5)
    )

# Prepare unique geolocation keys
geo_keys = geolocation_prepared[
    ["geolocation_zip_code_prefix", "geolocation_state"]
].drop_duplicates().copy()

geo_keys.columns = ["zip_prefix", "state"]

In [2813]:
# Match customers with geolocation
customer_geo_check = customer_keys.merge(
    geo_keys,
    on=["zip_prefix", "state"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

# Match sellers with geolocation
seller_geo_check = seller_keys.merge(
    geo_keys,
    on=["zip_prefix", "state"],
    how="left",
    indicator=True,
    validate="many_to_one"
)

print("CUSTOMER GEOLOCATION MATCHING")
print("Total customers:", len(customer_geo_check))
print("Matched:", (customer_geo_check["_merge"] == "both").sum())
print("Unmatched:", (customer_geo_check["_merge"] == "left_only").sum())

print("\nSELLER GEOLOCATION MATCHING")
print("Total sellers:", len(seller_geo_check))
print("Matched:", (seller_geo_check["_merge"] == "both").sum())
print("Unmatched:", (seller_geo_check["_merge"] == "left_only").sum())

CUSTOMER GEOLOCATION MATCHING
Total customers: 99441
Matched: 99162
Unmatched: 279

SELLER GEOLOCATION MATCHING
Total sellers: 3095
Matched: 3053
Unmatched: 42


### 4.6.3 Investigate Unmatched Customer and Seller Locations

Identify ZIP prefix and state combinations that do not
match the prepared geolocation reference.

Unmatched records will be retained and documented.

In [2814]:
# Extract unmatched customer locations

customer_unmatched = customer_geo_check.loc[
    customer_geo_check["_merge"] == "left_only",
    ["zip_prefix", "state"]
].copy()

# Extract unmatched seller locations

seller_unmatched = seller_geo_check.loc[
    seller_geo_check["_merge"] == "left_only",
    ["zip_prefix", "state"]
].copy()

# Count distinct unmatched location combinations

print("CUSTOMER LOCATIONS")
print("Unmatched records:", len(customer_unmatched))
print(
    "Distinct unmatched ZIP + state combinations:",
    customer_unmatched.drop_duplicates().shape[0]
)

print("\nSELLER LOCATIONS")
print("Unmatched records:", len(seller_unmatched))
print(
    "Distinct unmatched ZIP + state combinations:",
    seller_unmatched.drop_duplicates().shape[0]
)

CUSTOMER LOCATIONS
Unmatched records: 279
Distinct unmatched ZIP + state combinations: 158

SELLER LOCATIONS
Unmatched records: 42
Distinct unmatched ZIP + state combinations: 41


In [2815]:
print("Unmatched customer locations:")
display(
    customer_unmatched.value_counts()
    .reset_index(name="Record Count")
    .head(20)
)

print("Unmatched seller locations:")
display(
    seller_unmatched.value_counts()
    .reset_index(name="Record Count")
    .head(20)
)

Unmatched customer locations:


,zip_prefix,state,Record Count
0,70686,DF,15
1,72005,DF,13
2,71919,DF,10
3,73255,DF,7
4,72300,DF,6
5,73369,DF,5
6,71884,DF,5
7,72002,DF,5
8,73401,DF,5
9,71676,DF,4


Unmatched seller locations:


,zip_prefix,state,Record Count
0,88301,SP,2
1,95076,SP,1
2,21210,RN,1
3,85301,SP,1
4,87360,SP,1
5,86170,SP,1
6,85960,SP,1
7,22783,SP,1
8,36512,SP,1
9,82040,PR,1


### 4.6.4 Investigate Geolocation Matching Issues

Classify unmatched customer and seller locations
according to their possible causes.

Unmatched records will be retained in the project data.
No location will be assigned through guesswork.

In [2816]:
# Create sets of original and prepared geolocation keys

def create_location_keys(df, zip_col, state_col):

    temp = df[[zip_col, state_col]].copy()

    temp[zip_col] = (
        temp[zip_col].astype("string")
        .str.strip()
        .str.zfill(5)
    )

    temp[state_col] = (
        temp[state_col].astype("string")
        .str.strip()
        .str.upper()
    )

    return set(
        map(tuple, temp.dropna().itertuples(
            index=False, name=None
        ))
    )


# Geolocation keys before and after coordinate filtering
original_geo_keys = create_location_keys(
    geo_deduplicated,
    "geolocation_zip_code_prefix",
    "geolocation_state"
)

prepared_geo_keys = create_location_keys(
    geolocation_prepared,
    "geolocation_zip_code_prefix",
    "geolocation_state"
)

original_geo_zips = {
    zip_code for zip_code, state in original_geo_keys
}

print("Original ZIP + state combinations:", len(original_geo_keys))
print("Prepared ZIP + state combinations:", len(prepared_geo_keys))

Original ZIP + state combinations: 19023
Prepared ZIP + state combinations: 19018


In [2817]:
def classify_unmatched_locations(unmatched_df):

    locations = (
        unmatched_df.groupby(["zip_prefix", "state"])
        .size()
        .reset_index(name="Record Count")
    )

    def get_reason(row):

        key = (row["zip_prefix"], row["state"])

        if key in original_geo_keys:
            return "Removed by coordinate filter"

        elif row["zip_prefix"] in original_geo_zips:
            return "State mismatch"

        else:
            return "ZIP prefix absent from source"

    locations["Reason"] = locations.apply(
        get_reason, axis=1
    )

    summary = (
        locations.groupby("Reason")
        .agg(
            Distinct_Combinations=("zip_prefix", "size"),
            Affected_Records=("Record Count", "sum")
        )
        .reset_index()
    )

    return summary, locations


customer_geo_reasons, customer_geo_details = (
    classify_unmatched_locations(customer_unmatched)
)

seller_geo_reasons, seller_geo_details = (
    classify_unmatched_locations(seller_unmatched)
)

print("CUSTOMER MATCHING ISSUES")
display(customer_geo_reasons)

print("SELLER MATCHING ISSUES")
display(seller_geo_reasons)

CUSTOMER MATCHING ISSUES


,Reason,Distinct_Combinations,Affected_Records
0,Removed by coordinate filter,1,1
1,ZIP prefix absent from source,157,278


SELLER MATCHING ISSUES


,Reason,Distinct_Combinations,Affected_Records
0,State mismatch,34,35
1,ZIP prefix absent from source,7,7


### 4.6.5 Geolocation Preparation Results

Geolocation preparation:

- Original records: 1,000,163
- Exact duplicates removed: 261,831
- Records excluded by Brazil coordinate bounds: 38
- Prepared ZIP + state records: 19,018
- Duplicate ZIP + state keys: 0

Customer matching:
- Total customers: 99,441
- Matched: 99,162
- Unmatched: 279

Seller matching:
- Total sellers: 3,095
- Matched: 3,053
- Unmatched: 42

Unmatched records will be retained without assigning
unverified geographical coordinates.

ZIP prefix and state will be used together
as the geolocation matching key.

Original source data remains unchanged.

In [2818]:
geo_summary = pd.DataFrame([
    ["Original geolocation records", len(geo_df)],
    ["Exact duplicates removed", geo_df.duplicated().sum()],
    ["Records excluded by coordinate filter",
     len(geo_deduplicated) - len(geo_brazil)],
    ["Prepared ZIP + state records", len(geolocation_prepared)],
    ["Unmatched customer records", len(customer_unmatched)],
    ["Unmatched seller records", len(seller_unmatched)]
], columns=["Measurement", "Count"])

REPORT_DIR.mkdir(parents=True, exist_ok=True)

geo_summary.to_csv(
    REPORT_DIR / "04_geolocation_preparation_summary.csv",
    index=False
)

display(geo_summary)

print("Geolocation report saved successfully.")

,Measurement,Count
0,Original geolocation records,1000163
1,Exact duplicates removed,261831
2,Records excluded by coordinate filter,38
3,Prepared ZIP + state records,19018
4,Unmatched customer records,279
5,Unmatched seller records,42


Geolocation report saved successfully.


## 4.7 Final Data Preparation Validation

Combine the prepared working datasets into one final
collection for analysis.

Validate record counts, project scope, review selection,
product categories and geolocation keys.

The original CSV files remain unchanged.

In [2819]:
# Create the final prepared dataset collection

final_prepared_data = {
    "orders": scoped_orders.copy(),
    "order_items": prepared_data["order_items"].copy(),
    "order_payments": prepared_data["order_payments"].copy(),
    "order_reviews": latest_reviews.copy(),
    "customers": prepared_data["customers"].copy(),
    "sellers": prepared_data["sellers"].copy(),
    "products": products_prepared.copy(),
    "category_translation": translation_df.copy(),
    "geolocation": geolocation_prepared.copy()
}

print("Final prepared datasets:", len(final_prepared_data))

Final prepared datasets: 9


In [2820]:
# Validate final preparation

scope_ids = set(final_prepared_data["orders"]["order_id"])

# Check all order-related datasets are within project scope
assert (
    final_prepared_data["orders"]["order_id"].isin(scope_ids).all()
)

assert (
    final_prepared_data["order_items"]["order_id"].isin(scope_ids).all()
)

assert (
    final_prepared_data["order_payments"]["order_id"].isin(scope_ids).all()
)

assert (
    final_prepared_data["order_reviews"]["order_id"].isin(scope_ids).all()
)

# Check unique review per order
assert final_prepared_data["order_reviews"]["order_id"].is_unique

# Check product categories
assert final_prepared_data["products"]["analysis_category"].notna().all()

# Check geolocation keys
assert not final_prepared_data["geolocation"].duplicated(
    ["geolocation_zip_code_prefix", "geolocation_state"]
).any()

print("All final preparation checks passed!")

All final preparation checks passed!


In [2821]:
final_summary = pd.DataFrame([
    ["Project-period orders", len(scoped_orders)],
    ["K02 eligible orders", len(eligible_orders)],
    ["Valid delivery orders", len(delivery_orders)],
    ["Valid K07 orders", len(dispatch_kpi_df)],
    ["Latest reviews", len(latest_reviews)],
    ["Prepared products", len(products_prepared)],
    ["Unique product categories", products_prepared["analysis_category"].nunique()],
    ["Prepared geolocation records", len(geolocation_prepared)],
    ["Unmatched customer records", len(customer_unmatched)],
    ["Unmatched seller records", len(seller_unmatched)]
], columns=["Measurement", "Count"])

display(final_summary)

# Save the report
final_summary.to_csv(
    REPORT_DIR / "04_data_preparation_summary.csv",
    index=False
)

print("Final preparation report saved successfully.")

,Measurement,Count
0,Project-period orders,99092
1,K02 eligible orders,97910
2,Valid delivery orders,96203
3,Valid K07 orders,96202
4,Latest reviews,98330
5,Prepared products,32951
6,Unique product categories,74
7,Prepared geolocation records,19018
8,Unmatched customer records,279
9,Unmatched seller records,42


Final preparation report saved successfully.
